In [1]:
"""
task3 — extract signals with z-score normalization (all 3 windows)
-------------------------------------------------------------------
Computes signal_before, signal_after, signal_change, behavioral_improvement
using Z-SCORE NORMALIZATION before averaging features into a composite.

Why normalization matters:
  unlock_duration_ep_0 is in milliseconds (~15,000) while other digital_habits
  features are in single digits (~1–50). Without normalization, the composite
  is 99%+ determined by unlock_duration alone, making behavioral_improvement
  effectively mean only "did unlock_duration go down?", ignoring all other features.
  Z-score puts every feature on the same scale (mean=0, std=1) so all features
  contribute equally to the composite improvement decision.

Outputs (one file per window, preserving all prior columns):
  prompts_with_signals_task3_1day.csv
  prompts_with_signals_task3_3day.csv
  prompts_with_signals_task3_7day.csv

These filenames match what task4 expects — no downstream changes needed.
"""

import numpy as np
import pandas as pd
from datetime import timedelta

# ── Config ────────────────────────────────────────────────────────────────────
PROMPTS_CSV = "prompts_with_journal_responses_task2.csv"
FEATS_CSV   = "feat_dataset_filtered_with_weeks.csv"

WINDOWS = {
    "1day": 1,
    "3day": 3,
    "7day": 7,
}

# ── Domain → sensing columns ──────────────────────────────────────────────────
DOMAIN_COLS = {
    'digital_habits': [
        'app_Communication_ep_0',
        'app_Entertainment_ep_0',
        'app_Social_ep_0',
        'unlock_duration_ep_0',
        'unlock_num_ep_0',
    ],
    'social_interaction': [
        'call_in_duration_ep_0',  'call_out_duration_ep_0',
        'call_in_num_ep_0',       'call_out_num_ep_0',
        'loc_food_convo_duration','loc_food_convo_num', 'loc_food_still',
        'loc_greek_dur',          'loc_home_dur',
        'loc_home_convo_duration','loc_home_convo_num',
        'loc_other_dorm_dur',     'loc_self_dorm_dur',  'loc_study_dur',
        'sms_in_num_ep_0',        'sms_out_num_ep_0',
    ],
    'sleep': [
        'sleep_duration',
    ],
    'physical_fitness': [
        'loc_workout_dur',
        'act_on bike_ep_0',
        'act_on foot_ep_0',
        'act_walking_ep_0',
        'act_running_ep_0',
    ],
}

# Improvement direction per domain
# True  → higher composite signal = improvement
# False → lower  composite signal = improvement (e.g. less screen time = better)
HIGHER_IS_BETTER = {
    'physical_fitness':  True,
    'sleep':             True,
    'digital_habits':    False,   # lower screen/app time = better
    'social_interaction':True,
}

# ── Load data ─────────────────────────────────────────────────────────────────
print("Loading data...")
prompts_raw = pd.read_csv(PROMPTS_CSV)
feats       = pd.read_csv(FEATS_CSV, low_memory=False)

prompts_raw['date'] = pd.to_datetime(prompts_raw['date']).dt.date
feats['day']        = pd.to_datetime(feats['day']).dt.date

print(f"  Prompts : {len(prompts_raw)} rows")
print(f"  Feats   : {feats.shape[0]} rows | {feats.shape[1]} cols")

# Filter prompts — same logic as before
prompts_raw = prompts_raw[prompts_raw['behavioral_domain_category'] != 'general'].copy()
prompts_raw = prompts_raw[prompts_raw['journal_response_text'].notna()].copy()
prompts_raw = prompts_raw.reset_index(drop=True)
print(f"  Prompts after filtering: {len(prompts_raw)} rows")

# ── Step 1: Compute global z-score stats (mean, std) per feature ──────────────
# Stats computed across ALL users and ALL days — consistent scale.
print("\nComputing global z-score statistics per feature...")
all_feat_cols   = sorted(set(c for cols in DOMAIN_COLS.values() for c in cols))
valid_feat_cols = [c for c in all_feat_cols if c in feats.columns]
missing         = [c for c in all_feat_cols if c not in feats.columns]
if missing:
    print(f"  WARNING — missing columns (will be skipped): {missing}")

zscore_stats = {}
for col in valid_feat_cols:
    mean = feats[col].mean(skipna=True)
    std  = feats[col].std(skipna=True)
    zscore_stats[col] = (mean, std if std > 0 else 1.0)  # avoid div-by-zero

print(f"  Z-score stats computed for {len(zscore_stats)} features")


# ── Step 2: Helper — normalized window average ────────────────────────────────
def normalize(value, col):
    mean, std = zscore_stats[col]
    return (value - mean) / std


def window_avg_normalized(uid, center_date, col, direction, window):
    """
    Mean of z-scored daily values over `window` days before or after center_date.
    Returns (normalized_avg, raw_avg). Both NaN if no data in window.
    """
    if direction == 'before':
        dates = [center_date - timedelta(days=i) for i in range(1, window + 1)]
    else:
        dates = [center_date + timedelta(days=i) for i in range(1, window + 1)]

    subset = feats[(feats['uid'] == uid) & (feats['day'].isin(dates))][col]
    subset = pd.to_numeric(subset, errors='coerce').dropna()

    if subset.empty:
        return np.nan, np.nan

    raw_avg  = subset.mean()
    norm_avg = normalize(raw_avg, col)
    return norm_avg, raw_avg


# ── Step 3: Process each window ───────────────────────────────────────────────
for window_label, window_days in WINDOWS.items():
    print(f"\n{'='*60}")
    print(f"Processing window: {window_label} ({window_days}-day before/after)")
    print(f"{'='*60}")

    prompts = prompts_raw.copy()
    main_rows = []

    for i, row in prompts.iterrows():
        domain = row['behavioral_domain_category']
        uid    = row['participant_id']
        jdate  = row['date']

        if domain not in DOMAIN_COLS:
            main_rows.append({
                'signal_before': np.nan, 'signal_after': np.nan,
                'signal_change': np.nan, 'behavioral_improvement': np.nan,
                'n_features_total': np.nan, 'n_features_improved': np.nan,
                'baseline_imputed': np.nan,
            })
            continue

        higher    = HIGHER_IS_BETTER[domain]
        feat_cols = [c for c in DOMAIN_COLS[domain] if c in feats.columns]

        b_norms, a_norms, n_improved = [], [], 0

        for col in feat_cols:
            b_norm, _ = window_avg_normalized(uid, jdate, col, 'before', window_days)
            a_norm, _ = window_avg_normalized(uid, jdate, col, 'after',  window_days)

            if pd.isna(b_norm) or pd.isna(a_norm):
                continue

            change_norm   = a_norm - b_norm
            feat_improved = bool((higher and change_norm > 0) or (not higher and change_norm < 0))

            b_norms.append(b_norm)
            a_norms.append(a_norm)
            if feat_improved:
                n_improved += 1

        # Composite = mean of normalized feature values
        # NaN windows → 0 (global mean in z-score space, conservative imputation)
        comp_b   = np.mean(b_norms) if b_norms else 0.0
        comp_a   = np.mean(a_norms) if a_norms else 0.0
        comp_c   = comp_a - comp_b
        improved = bool((higher and comp_c > 0) or (not higher and comp_c < 0))
        imputed  = (not b_norms) or (not a_norms)

        main_rows.append({
            'signal_before':          round(comp_b, 4),
            'signal_after':           round(comp_a, 4),
            'signal_change':          round(comp_c, 4),
            'behavioral_improvement': 'yes' if improved else 'no',
            'n_features_total':       len(feat_cols),
            'n_features_improved':    n_improved,
            'baseline_imputed':       'yes' if imputed else 'no',
        })

    # Attach new columns
    new_cols    = pd.DataFrame(main_rows, index=prompts.index)
    prompts_out = pd.concat([prompts, new_cols], axis=1)

    # Summary
    print(f"  Total rows            : {len(prompts_out)}")
    print(f"  Improvement breakdown : {prompts_out['behavioral_improvement'].value_counts().to_dict()}")
    print(f"\n  By domain:")
    by_domain = (
        prompts_out[prompts_out['behavioral_improvement'].notna()]
        .groupby(['behavioral_domain_category', 'behavioral_improvement']).size()
        .unstack(fill_value=0)
    )
    print(by_domain.to_string())

    # Export — same filename pattern task4 already reads
    out_file = f"prompts_with_signals_task3_{window_label}.csv"
    prompts_out.to_csv(out_file, index=False)
    print(f"\n  Saved → {out_file}  ({len(prompts_out)} rows, {len(prompts_out.columns)} cols)")

print("\nAll windows complete. task4 can now read these files with no changes.")


Loading data...
  Prompts : 648 rows
  Feats   : 1072 rows | 523 cols
  Prompts after filtering: 369 rows

Computing global z-score statistics per feature...
  Z-score stats computed for 27 features

Processing window: 1day (1-day before/after)
  Total rows            : 369
  Improvement breakdown : {'no': 205, 'yes': 164}

  By domain:
behavioral_improvement      no  yes
behavioral_domain_category         
digital_habits              77   51
physical_fitness            43   42
sleep                       22   16
social_interaction          63   55

  Saved → prompts_with_signals_task3_1day.csv  (369 rows, 13 cols)

Processing window: 3day (3-day before/after)
  Total rows            : 369
  Improvement breakdown : {'no': 200, 'yes': 169}

  By domain:
behavioral_improvement      no  yes
behavioral_domain_category         
digital_habits              74   54
physical_fitness            42   43
sleep                       22   16
social_interaction          62   56

  Saved → prompts_wi